# Exploratory Data Analysis
## Paycheck Protection Program (PPP) Loan Fraud Analysis

## Goal
The goal of this notebook is to understand the structure of the PPP dataset before cleaning.

**Dataset:** SBA PPP Public Data - loans above $150,000  
**Period:** 2020-2021  
**Source:** Small Business Administration (sba.gov)

In [5]:
import pandas as pd

RAW_FILE = '../data/raw/public_150k_plus_240930.csv'
df = pd.read_csv(RAW_FILE, encoding='latin-1')

## 0. Data dictionary - column selection
Before exploring the data, the official SBA data dicionary is reviewed to identify which columns are relevant for fraud analysis.

In [6]:
data_dict = pd.read_excel('../data/raw/ppp-data-dictionary.xlsx')
display(data_dict)

,Field Name,Field Description
0,LoanNumber,Loan Number (unique identifier)
1,DateApproved,Loan Funded Date
2,SBAOfficeCode,SBA Origination Office Code
3,ProcessingMethod,Loan Delivery Method (PPP for first draw; PPS ...
4,BorrowerName,Borrower Name
5,BorrowerAddress,Borrower Street Address
6,BorrowerCity,Borrower City
7,BorrowerState,Borrower State
8,BorrowerZip,Borrower Zip Code
9,LoanStatusDate,Loan Status Date\n- Loan Status Date is blank...


### Columns to KEEP — relevant for fraud analysis:
- DateApproved, BorrowerName, BorrowerAddress, BorrowerCity, BorrowerState, BorrowerZip
- LoanStatus, LoanStatusDate
- InitialApprovalAmount, CurrentApprovalAmount
- JobsReported
- BusinessType, BusinessAgeDescription
- NAICSCode
- OriginatingLender, OriginatingLenderCity, OriginatingLenderState
- ServicingLenderName
- ForgivenessAmount, ForgivenessDate
- ProcessingMethod, ProjectState
- PAYROLL_PROCEED

### Columns to DROP — not relevant for fraud analysis:
- SBAOfficeCode, SBAGuarantyPercentage, internal SBA codes, same for all loans
- ServicingLenderLocationID, ServicingLenderAddress, ServicingLenderCity, ServicingLenderState, ServicingLenderZip, bank address details, not needed
- OriginatingLenderLocationID, internal ID, have lender name already
- ProjectCity, ProjectCountyName, ProjectZip, CD, redundant location data
- Race, Ethnicity, Gender, Veteran, demographic data, not fraud indicators
- RuralUrbanIndicator, HubzoneIndicator, LMIIndicator, location indicators, not fraud indicators
- LoanNumber, unique identifier, not needed for analysis
- Term, same for all PPP loans
- UndisbursedAmount, internal banking data, not fraud indicator
- FranchiseName, NonProfit, UTILITIES_PROCEED, MORTGAGE_INTEREST_PROCEED, RENT_PROCEED, REFINANCE_EIDL_PROCEED, HEALTH_CARE_PROCEED, DEBT_INTEREST_PROCEED, proceed fields, not needed for fraud detection

## 1. Basic dataset structure
How many rows and columns does the dataset have? What columns exist?

In [7]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
display(pd.DataFrame(df.columns, columns=['Column name']))

Rows: 968,524
Columns: 53


,Column name
0,LoanNumber
1,DateApproved
2,SBAOfficeCode
3,ProcessingMethod
4,BorrowerName
5,BorrowerAddress
6,BorrowerCity
7,BorrowerState
8,BorrowerZip
9,LoanStatusDate


## 2. First 10 rows
Visual inspection of what the raw data looks like

In [8]:
display(df.head(10))

,LoanNumber,DateApproved,SBAOfficeCode,ProcessingMethod,BorrowerName,BorrowerAddress,BorrowerCity,BorrowerState,BorrowerZip,LoanStatusDate,...,BusinessType,OriginatingLenderLocationID,OriginatingLender,OriginatingLenderCity,OriginatingLenderState,Gender,Veteran,NonProfit,ForgivenessAmount,ForgivenessDate
0,9547507704,05/01/2020,464,PPP,"SUMTER COATINGS, INC.",2410 Highway 15 South,Sumter,NaN,29150-9662,12/18/2020,...,Corporation,19248,Synovus Bank,COLUMBUS,GA,Unanswered,Unanswered,NaN,773553.37,11/20/2020
1,9777677704,05/01/2020,464,PPP,"PLEASANT PLACES, INC.",7684 Southrail Road,North Charleston,NaN,29420-9000,09/28/2021,...,Sole Proprietorship,19248,Synovus Bank,COLUMBUS,GA,Male Owned,Non-Veteran,NaN,746336.24,08/12/2021
2,5791407702,05/01/2020,1013,PPP,BOYER CHILDREN'S CLINIC,1850 BOYER AVE E,SEATTLE,NaN,98112-2922,03/17/2021,...,Non-Profit Organization,9551,"Bank of America, National Association",CHARLOTTE,NC,Unanswered,Unanswered,Y,696677.49,02/10/2021
3,6223567700,05/01/2020,920,PPP,KIRTLEY CONSTRUCTION INC,1661 MARTIN RANCH RD,SAN BERNARDINO,NaN,92407-1740,10/16/2021,...,Corporation,9551,"Bank of America, National Association",CHARLOTTE,NC,Female Owned,Non-Veteran,NaN,395264.11,09/10/2021
4,9662437702,05/01/2020,101,PPP,AERO BOX LLC,NaN,NaN,NaN,NaN,08/17/2021,...,NaN,57328,The Huntington National Bank,COLUMBUS,OH,Unanswered,Unanswered,NaN,370819.35,04/08/2021
5,9774337701,05/01/2020,101,PPP,HUDSON EXTRUSIONS INC.,NaN,NaN,NaN,NaN,11/17/2021,...,NaN,57328,The Huntington National Bank,COLUMBUS,OH,Unanswered,Unanswered,NaN,332137.41,05/10/2021
6,9794577700,05/01/2020,491,PPP,FRUIT COVE BAPTIST CHURCH OF JACKSONVILLE FL INC,501 State Road 13,Saint Johns,NaN,32259-2832,02/19/2021,...,Non-Profit Organization,19248,Synovus Bank,COLUMBUS,GA,Unanswered,Unanswered,Y,291741.75,01/07/2021
7,9722187702,05/01/2020,101,PPP,MIAMITOWN AUTO PARTS AND RECYCLING INC,NaN,NaN,NaN,NaN,02/24/2021,...,Corporation,57328,The Huntington National Bank,COLUMBUS,OH,Unanswered,Unanswered,NaN,269416.44,01/25/2021
8,9725917702,05/01/2020,101,PPP,POPPYCOCKS INC,NaN,NaN,NaN,NaN,08/17/2021,...,Subchapter S Corporation,57328,The Huntington National Bank,COLUMBUS,OH,Unanswered,Unanswered,NaN,259982.88,06/24/2021
9,9666867710,05/01/2020,101,PPP,CHURCH SQUARE PHARMACY INC,NaN,NaN,NaN,NaN,04/21/2021,...,Corporation,57328,The Huntington National Bank,COLUMBUS,OH,Unanswered,Unanswered,NaN,252253.42,03/31/2021


## 3. Data types
Checking whether columns are correctly recognized - numeric, text, dates

In [9]:
display(df.dtypes.reset_index().rename(columns={'index':'Column',0:'Data Type'}))

,Column,Data Type
0,LoanNumber,int64
1,DateApproved,object
2,SBAOfficeCode,int64
3,ProcessingMethod,object
4,BorrowerName,object
5,BorrowerAddress,object
6,BorrowerCity,object
7,BorrowerState,object
8,BorrowerZip,object
9,LoanStatusDate,object


## 4. Missing values
Which columns have missing values and what percentage?

In [10]:
null_percent = (df.isnull().sum() / len(df) * 100).round(1)
null_df = null_percent[null_percent > 0].sort_values(ascending=False).reset_index()
null_df.columns = ['Column', '% Missing']
display(null_df)

,Column,% Missing
0,REFINANCE_EIDL_PROCEED,97.6
1,DEBT_INTEREST_PROCEED,96.7
2,FranchiseName,96.3
3,MORTGAGE_INTEREST_PROCEED,95.2
4,NonProfit,94.1
5,HEALTH_CARE_PROCEED,94.1
6,RENT_PROCEED,89.7
7,UTILITIES_PROCEED,65.0
8,ForgivenessDate,2.6
9,ForgivenessAmount,2.6


### Finding:
8 columns have extremely high missing value rates and will be dropped in the ETL phase:

REFINANCE_EIDL_PROCEED (97.6%), DEBT_INTEREST_PROCEED (96.7%), FranchiseName (96.3%),
MORTGAGE_INTEREST_PROCEED (95.2%), NonProfit (94.1%), HEALTH_CARE_PROCEED (94.1%),
RENT_PROCEED (89.7%), UTILITIES_PROCEED (65.0%).

## 5. Column LoanStatus and LoanStatusDate - Loan status
What values exist? How many loans are charged off?

In [23]:
loan_status = df['LoanStatus'].value_counts().reset_index()
loan_status.columns = ['Status', 'Count']
loan_status['%'] = (loan_status['Count'] / len(df) * 100).round(1)
display(loan_status)

loan_date = df['LoanStatusDate'].isna().sum()
display(loan_date)

,Status,Count,%
0,Paid in Full,940091,97.1
1,Charged Off,17105,1.8
2,Exemption 4,11328,1.2


np.int64(11328)

### Finding:
97.1% of loans are Paid in Full, expected for a forgiveness program.

17,105 loans (1.8%) are Charged Off - the borrower never repaid and the loan was written off as a loss. These are primary fraud candidates.

11,328 loans (1.2%) show Exemption 4 status, meaning the SBA withheld their loan status under FOIA Exemption 4, which protects confidential business and financial information. These loans were disbursed but are neither Paid in Full nor Charged Off, their outcome is unknown due to missing data.

LoanStatusDate null values match the 11,328 Exemption 4 loans in LoanStatus,
confirming that these two columns consistently identify the same loans -
disbursed but with unknown outcome.

## 6. Column InitialApprovalAmount - Loan Amount
What is the distribution of loan amounts? Are there extreme values (outliers)?

In [12]:
amount = df['InitialApprovalAmount']

print(f'Minimum loan: $ {amount.min():>15,.0f}')
print(f'Maximum loan: $ {amount.max():>15,.0f}')
print(f'Average loan: $ {amount.mean():>15,.0f}')
print(f'Median loan:  $ {amount.median():>15,.0f}')

Minimum loan: $               0
Maximum loan: $      10,000,000
Average loan: $         532,272
Median loan:  $         295,177


### Finding:
Maximum loan is $10,000,000, the offical program upper limit, expected.

Minimum loan is $0, this needs investigation in the ETL phase.
A loan of $0 makes no sense and likely indicates a data entry error.

The average ($532,272) is significantly higher than the median ($295,177),
indicating a right-skewed distribution, a small number of very large loans
are pulling the average up. This is expected since larger companies
naturally require more funding to cover payroll.

## 7. Column JobsReported - number of reported employees
This is the key fraud column. Companies that took large loans  
but reported 0 or very few employees are the main suspects.

In [13]:
jobs = df['JobsReported']

print(f'Minimum:     {jobs.min():>10,.0f}')
print(f'Maximum:     {jobs.max():>10,.1f}')
print(f'Average:     {jobs.mean():>10,.1f}')
print(f'Median:      {jobs.median():>10,.1f}')
print(f'Null values: {jobs.isnull().sum():>10,}')
print(f'Zero values: {(jobs == 0).sum():>10,}')

Minimum:              0
Maximum:          500.0
Average:           51.9
Median:            30.0
Null values:          1
Zero values:          4


### Finding:
Only 4 loans reported 0 employees and 1 has a null value, surprisingly low.
Zero employees alone is not necessarily fraud, self-employed individuals 
and sole proprietors were eligible for PPP loans.
However, 0 employees combined with a large loan amount is a strong fraud indicator and will be flagged in the risk scoring model.

Maximum reported employees is 500, the official program limit.

Average is 51.9 and median is 30, indicating most borrowers were small businesses.

Loan amount must always be analyzed in combination with JobsReported
to identify disproportionate loans.

## 8. Column DateApproved - approval date
What is the time range of the dataset?

In [14]:
print(f"Earliest loan: {df['DateApproved'].min()}")
print(f"Latest loan:   {df['DateApproved'].max()}")
print()
print("Distribution by year:")
display(df['DateApproved'].str[:4].value_counts().sort_index().reset_index().rename(
    columns={'index': 'Year', 'DateApproved': 'Count'}))

Earliest loan: 01/13/2021
Latest loan:   12/14/2020

Distribution by year:


,Count,count
0,01/1,11148
1,01/2,81263
2,01/3,12743
3,02/0,47305
4,02/1,40158
5,02/2,24873
6,03/0,10995
7,03/1,29013
8,03/2,20205
9,03/3,5926


## 9. Column OriginatingLender - banks
Which banks approved the most loans?

In [15]:
lenders = df['OriginatingLender'].value_counts().head(15).reset_index()
lenders.columns = ['Bank', 'Loan Count']
display(lenders)

,Bank,Loan Count
0,"JPMorgan Chase Bank, National Association",57221
1,"Bank of America, National Association",40259
2,Truist Bank,20954
3,"PNC Bank, National Association",20527
4,Manufacturers and Traders Trust Company,17118
5,"Wells Fargo Bank, National Association",15459
6,"TD Bank, National Association",15396
7,The Huntington National Bank,14212
8,KeyBank National Association,13442
9,"Zions Bank, A Division of",12694


## 10. Column BusinessType - type of business
What types of businesses exist in the dataset?

In [16]:
business = df['BusinessType'].value_counts(dropna=False).reset_index()
business.columns = ['Business Type', 'Count']
display(business)

,Business Type,Count
0,Corporation,418229
1,Limited Liability Company(LLC),261498
2,Subchapter S Corporation,174794
3,Non-Profit Organization,55950
4,Partnership,18150
5,Limited Liability Partnership,12708
6,Sole Proprietorship,12242
7,Professional Association,6253
8,Cooperative,2433
9,501(c)3 ¿ Non Profit,1777


## 11. Column ForgivenessAmount - How much was forgiven
How many loands were never forgiven? There are potential fraud cases.

In [17]:
forgivness = df['ForgivenessAmount']

print(f'Minimum:                    ${forgivness.min():>15,.0f}')
print(f'Maximum:                    ${forgivness.max():>15,.0f}')
print(f'Average:                    ${forgivness.mean():>15,.0f}')
print(f'Median:                     ${forgivness.median():>15,.0f}')
print(f'Null (never forgiven):      ${forgivness.isnull().sum():>10,}')
print()
print('Loans where forgiveness = 0:')
print((forgivness == 0).sum())

Minimum:                    $              0
Maximum:                    $     10,361,644
Average:                    $        529,392
Median:                     $        295,598
Null (never forgiven):      $    25,463

Loans where forgiveness = 0:
0


### Finding:
25,463 loans have no forgiveness amount recorded (null), these borrowers 
never applied for forgiveness, meaning the loan was never officially cleared.
This is a significant fraud indicator, legitimate borrowers had strong 
incentive to apply for forgiveness since it meant free money.

No loans have forgiveness amount of exactly $0, all recorded forgiveness 
amounts are greater than zero, which is expected.

Maximum forgiveness amount is $10,361,644, slightly above the $10M loan limit,
which needs investigation in ETL.

The average forgiveness ($529,392) closely matches the average loan amount ($532,272),
suggesting that most approved loans were fully forgiven, consistent with 
program design.

## 12. Column ForgivenessDate - when were loans forgiven
Are there loans approved but never assigned a forgiveness date?

In [18]:
print(f"Missing forgiveness date: {df['ForgivenessDate'].isnull().sum():,}")
print(f"Total with forgiveness date: {df['ForgivenessDate'].notnull().sum()}")

Missing forgiveness date: 25,463
Total with forgiveness date: 943061


### Finding:
There are 25,463 missing forgiveness dates, indicating that those companies 
never applied for loan forgiveness.

This is a strong fraud indicator, borrowers had a clear financial incentive 
to apply for forgiveness since it meant free money. Legitimate businesses 
rarely forgo this opportunity.

This number matches exactly with the 25,463 null values in ForgivenessAmount column,
confirming that these loans were never forgiven and will be flagged 
as potential fraud candidates in the risk scoring model.

## 13. Column BorrowerAddress - company address
Legitimate businesses have a physical location

In [19]:
print(f'Missing address: {df["BorrowerAddress"].isnull().sum():,}')
print(f'Missing city: {df["BorrowerCity"].isnull().sum():,}')
print(f'Missing state: {df["BorrowerState"].isnull().sum():,}')
print(f'Missing zip: {df["BorrowerZip"].isnull().sum():,}')
print()
print('Loans with no adress at all: ')
no_adress = df[df['BorrowerAddress'].isnull() & df['BorrowerCity'].isnull()]
print(f'{len(no_adress):,} loans')
print()
display(no_adress[['BorrowerName', 'InitialApprovalAmount', 'JobsReported', 'LoanStatus']])

Missing address: 14
Missing city: 12
Missing state: 13
Missing zip: 13

Loans with no adress at all: 
12 loans



,BorrowerName,InitialApprovalAmount,JobsReported,LoanStatus
4,AERO BOX LLC,367437.0,25.0,Paid in Full
5,HUDSON EXTRUSIONS INC.,328840.0,22.0,Paid in Full
7,MIAMITOWN AUTO PARTS AND RECYCLING INC,272380.0,19.0,Paid in Full
8,POPPYCOCKS INC,257088.0,18.0,Paid in Full
9,CHURCH SQUARE PHARMACY INC,250000.0,17.0,Paid in Full
10,"MILFAST INDUSTRIAL SUPPLY, INC",174046.0,18.0,Paid in Full
11,FERNANDINA BEACH HOTEL GROUP LLC,170170.0,12.0,Paid in Full
12,OTTAWA PRODUCTS CO INC,155010.0,11.0,Paid in Full
158723,ROMUALD Z JOVERO DBA R &AMP; B INTERNATIONAL O...,158640.0,24.0,Paid in Full
665654,PETREY AB CORP.,189342.0,25.0,Charged Off


### Finding:
14 loans have missing address, 13 have missing city, state and zip values.
12 companies have no address information at all — no street, city, state or zip.

While missing address is a fraud indicator, legitimate businesses must have 
a physical location, the number is very small (12 out of 968,524 total loans, 
less than 0.001%) and unlikely to significantly impact the overall analysis.

These 12 companies will be investigated in combination with JobsReported 
and LoanStatus columns in the risk scoring phase to determine 
if they are potential fraud candidates.

## 14. Column BusinessAgeDescription - how old is the company
PPP was designed for existing businesses. New companies formed just before PPP are a major fraud red flag.

In [20]:
age = df['BusinessAgeDescription'].value_counts(dropna=False).reset_index()
age.columns = ['Business Age', 'Count']
age['%'] = (age['Count'] / len(df) * 100).round(1)
display(age)

,Business Age,Count,%
0,Existing or more than 2 years old,861909,89.0
1,New Business or 2 years or less,54851,5.7
2,Unanswered,51051,5.3
3,Change of Ownership,422,0.0
4,"Startup, Loan Funds will Open Business",290,0.0
5,NaN,1,0.0


### Finding:
89% of businesses are older than 2 years, indicating lower fraud risk 
for the majority of borrowers.

5.7% (54,851) are new businesses (2 years or less). PPP was designed for 
existing businesses, new companies formed just before COVID are a known 
fraud pattern and will be flagged in the risk scoring model.

5.3% (51,051) answered "Unanswered", this is concerning since legitimate 
businesses have no reason to hide their age. This group will be 
investigated in combination with JobsReported and LoanStatus.

1 null value exists and will be handled in the ETL phase.

## EDA Summary

### Dataset Overview
The SBA PPP Public Data dataset (loans above $150,000) contains 968,524 rows 
and 53 columns. The dataset is usable but requires cleaning before analysis.

### Issues to Resolve in ETL Phase
1. Columns to drop, 30 columns identified in Data Dictionary review as not relevant 
   for fraud analysis, including demographic data, internal IDs, bank address details,
   redundant location fields and proceed fields with high missing rates
2. Date columns, DateApproved, LoanStatusDate and ForgivenessDate 
   are stored as strings and must be converted to datetime format
3. InitialApprovalAmount, contains $0 values that need investigation
4. BusinessType, 713 null values, will be filled with 'Unknown'
5. NAICSCode, 0.7% missing, will be filled with 'Unknown'
6. BusinessAgeDescription, 1 null value, will be filled with 'Unknown'

### Potential Fraud Indicators Observed
1. 17,105 Charged Off loans, borrowers never repaid, loan written off as loss
2. 11,328 Exemption 4 loans, loan outcome unknown due to missing data,
   cannot be cleared as legitimate
3. 25,463 loans with no forgiveness, borrowers never applied for forgiveness
   despite having strong financial incentive to do so
4. 4 loans with 0 employees, suspicious when combined with large loan amounts
5. 12 companies with no address, legitimate businesses must have 
   a physical location
6. 54,851 new businesses (5.7%), companies formed just before COVID, 
   a known fraud pattern
7. 51,051 unanswered BusinessAgeDescription (5.3%), legitimate companies 
   have no reason to hide their age

### Next Step
Proceed to ETL phase, clean and prepare the dataset for SQL analysis.